# Phase 5 — Feature Engineering

This phase converts product titles into consistent categories and product specifications. The source file remains unchanged. The shared parser in `backend/product_features.py` is used here and by the live application, so training and live inference use the same category rules and title extraction.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from backend.product_features import classify_product, extract_title_specs, product_feature_frame

DATA_PATH = Path("../electronics_analysis_ready.csv")
OUTPUT_PATH = Path("electronics_feature_engineered.csv")
PHASE_OUTPUT_DIR = Path("phase_outputs")
PHASE_OUTPUT_DIR.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH).copy()
print("Source shape:", df.shape)
print("Source rows retained:", len(df))
print("Source columns:", df.columns.tolist())

Source shape: (5010, 17)
Source rows retained: 5010
Source columns: ['name', 'main_category', 'sub_category', 'image', 'link', 'no_of_ratings', 'discount_price', 'actual_price', 'review_rating', 'review_text', 'discount_price_missing', 'actual_price_missing', 'review_count_missing', 'review_text_missing', 'price_available', 'discount_amount', 'discount_percent']


## Product categories

Rules inspect accessory identity before compatible device brands. A phone title beginning with a known model stays a smartphone even if the listing mentions an included charger. Headphones are recognized before phone-maker prefixes, which prevents titles such as OnePlus Nord Buds from becoming smartphones. A title counts as a laptop only when it names a laptop **and** lists a processor plus memory, storage or an operating system; this is checked first, so a laptop that mentions DDR5 RAM, a GPU or a battery is not mistaken for a component. Bags, tables, stands, mice and cleaning kits that only mention "laptop" as a compatible device are accessories. Unclear products remain explicitly unclassified.

In [2]:
df["product_category"] = df["name"].map(classify_product)
category_counts = df["product_category"].value_counts().rename_axis("product_category").to_frame("count")
display(category_counts)
print("Total classified:", int((df.product_category != "Other / Unclassified").sum()))
print("Unclassified:", int((df.product_category == "Other / Unclassified").sum()))
print("Non-electronic / miscellaneous:", int((df.product_category == "Non-electronic / Miscellaneous").sum()))

unclassified = df.loc[df.product_category.eq("Other / Unclassified"), ["name"]]
print("Representative unclassified titles (first 30, source order):")
display(unclassified.head(30))

,count
product_category,
Other / Unclassified,1076
Headphones and Earphones,592
Mobile Accessories,532
Computer Accessories,493
Chargers and Cables,486
Smartphones,345
Smartwatches,214
Batteries,183
Tablets,140


Total classified: 3934
Unclassified: 1076
Non-electronic / miscellaneous: 120
Representative unclassified titles (first 30, source order):


,name
33,"sandisk ultra® microsdxc™ uhs-i card, 64gb, 14..."
38,samsung 25w usb travel adapter for cellular ph...
50,"jk copier paper - a4, 75 gsm, 1 ream, 500 sheets"
53,scotch double sided heavy duty tape(1m holds 4...
56,striff adjustable laptop tabletop stand patent...
81,"sandisk ultra® microsdxc™ uhs-i card, 128gb, 1..."
84,"mi cordless beard trimmer 1c, with 20 length s..."
90,classmate octane neon- blue gel pens(pack of 5...
104,samsung original 25w usb travel lightning adap...
115,hp 680 original ink advantage cartridge (black)


## Extract specifications and review representative records

The six values are title-derived features, so a blank value means the title did not provide enough evidence. Category gates prevent screen sizes and refresh rates from being extracted from unrelated products. These checks cover common title patterns and confirm that accessories and compatible-device mentions do not override the product identity.

In [3]:
spec_columns = ["ram_gb", "storage_gb", "battery_mah", "screen_inches", "charging_w", "refresh_rate_hz"]
specs = pd.DataFrame(
    [extract_title_specs(row.name, row.product_category) for row in df.itertuples()],
    index=df.index,
)
# itertuples exposes the title in the `name` field; make the assignment explicit.
df[spec_columns] = specs[spec_columns]

# Curated examples drawn from the source data and inspected for title meaning.
checks = [
    (r"^redmi 10 power", "Smartphones", "ram_gb", 8.0),
    (r"^oneplus nord ce 2 lite", "Smartphones", "storage_gb", 128.0),
    (r"^oneplus bullets z2", "Headphones and Earphones", None, None),
    (r"^mi lithium ion xiaomi wireless power bank", "Power Banks", "battery_mah", 10000.0),
    (r"^hp 14s, 5th gen", "Laptops", "storage_gb", 512.0),
    (r"^apple 20w usb-c power adapter", "Chargers and Cables", "charging_w", 20.0),
    (r"^spigen tough armor back cover", "Mobile Accessories", None, None),
    (r"^samsung galaxy m53 5g", "Smartphones", "ram_gb", 6.0),
    (r"^iphone original 20w charger", "Chargers and Cables", "charging_w", 20.0),
    (r"^gizga essentials universal silicone keyboard protector", "Computer Accessories", "screen_inches", np.nan),
    (r"^hp 250 g8 laptop", "Laptops", "ram_gb", 8.0),
    (r"^wesley milestone 2.0 casual waterproof laptop backpack", "Computer Accessories", "screen_inches", np.nan),
    (r"^logitech b100 wired usb mouse", "Computer Accessories", None, None),
]
check_rows = []
for pattern, expected_category, feature, expected_value in checks:
    match = df[df.name.str.contains(pattern, case=False, regex=True, na=False)]
    if match.empty:
        check_rows.append({"pattern": pattern, "found": False, "expected_category": expected_category, "actual_category": None, "feature": feature, "expected": expected_value, "actual": None, "pass": False})
        continue
    row = match.iloc[0]
    actual_value = row[feature] if feature else None
    value_ok = feature is None or (pd.isna(expected_value) and pd.isna(actual_value)) or (pd.notna(actual_value) and pd.notna(expected_value) and np.isclose(actual_value, expected_value))
    ok = row.product_category == expected_category and value_ok
    check_rows.append({"pattern": pattern, "found": True, "expected_category": expected_category, "actual_category": row.product_category, "feature": feature, "expected": expected_value, "actual": actual_value, "pass": bool(ok)})
checks_df = pd.DataFrame(check_rows)
display(checks_df)
assert checks_df["pass"].all(), "Review the specific category/spec extraction examples before proceeding."

coverage = (df[spec_columns].notna().mean() * 100).rename("coverage_percent").round(2).to_frame()
display(coverage)
print("Mean coverage across six specification features:", round(float(coverage.coverage_percent.mean()), 2), "%")
print("Missing values are retained; no records are removed.")

,pattern,found,expected_category,actual_category,feature,expected,actual,pass
0,^redmi 10 power,True,Smartphones,Smartphones,ram_gb,8.0,8.0,True
1,^oneplus nord ce 2 lite,True,Smartphones,Smartphones,storage_gb,128.0,128.0,True
2,^oneplus bullets z2,True,Headphones and Earphones,Headphones and Earphones,None,NaN,NaN,True
3,^mi lithium ion xiaomi wireless power bank,True,Power Banks,Power Banks,battery_mah,10000.0,10000.0,True
4,"^hp 14s, 5th gen",True,Laptops,Laptops,storage_gb,512.0,512.0,True
5,^apple 20w usb-c power adapter,True,Chargers and Cables,Chargers and Cables,charging_w,20.0,20.0,True
6,^spigen tough armor back cover,True,Mobile Accessories,Mobile Accessories,None,NaN,NaN,True
7,^samsung galaxy m53 5g,True,Smartphones,Smartphones,ram_gb,6.0,6.0,True
8,^iphone original 20w charger,True,Chargers and Cables,Chargers and Cables,charging_w,20.0,20.0,True
9,^gizga essentials universal silicone keyboard ...,True,Computer Accessories,Computer Accessories,screen_inches,NaN,NaN,True


,coverage_percent
ram_gb,4.89
storage_gb,5.25
battery_mah,2.95
screen_inches,3.37
charging_w,4.39
refresh_rate_hz,0.80


Mean coverage across six specification features: 3.61 %
Missing values are retained; no records are removed.


In [4]:
# Validate plausibility ranges and inspect a fixed sample of extracted rows.
ranges = {
    "ram_gb": (1, 128), "storage_gb": (1, 8192), "battery_mah": (100, 100000),
    "screen_inches": (1, 100), "charging_w": (1, 300), "refresh_rate_hz": (30, 360),
}
validation = []
for col, (lo, hi) in ranges.items():
    values = df[col].dropna()
    invalid = int((~values.between(lo, hi)).sum())
    validation.append({"feature": col, "available": int(values.size), "invalid_out_of_range": invalid})
validation_df = pd.DataFrame(validation)
display(validation_df)
assert validation_df.invalid_out_of_range.sum() == 0

category_coverage = df.groupby("product_category")[spec_columns].apply(lambda x: x.notna().mean() * 100).round(1)
plt.figure(figsize=(12, 7))
plt.imshow(category_coverage, aspect="auto", cmap="YlGnBu", vmin=0, vmax=100)
plt.colorbar(label="Coverage (%)")
plt.xticks(range(len(spec_columns)), spec_columns, rotation=35, ha="right")
plt.yticks(range(len(category_coverage)), category_coverage.index)
plt.title("Specification Coverage by Inferred Product Category")
plt.tight_layout()
plt.savefig(PHASE_OUTPUT_DIR / "phase5_feature_coverage.png", dpi=180, bbox_inches="tight")
plt.show()

display(df.loc[df[spec_columns].notna().any(axis=1), ["name", "product_category", *spec_columns]].sample(20, random_state=42))

,feature,available,invalid_out_of_range
0,ram_gb,245,0
1,storage_gb,263,0
2,battery_mah,148,0
3,screen_inches,169,0
4,charging_w,220,0
5,refresh_rate_hz,40,0


/var/folders/_y/jrtd4g7n1xq8nyb6qbq1yywh0000gn/T/ipykernel_98253/163902262.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,name,product_category,ram_gb,storage_gb,battery_mah,screen_inches,charging_w,refresh_rate_hz
4756,"flix usb charger,flix (beetel) bolt 2.4 dual p...",Chargers and Cables,NaN,NaN,NaN,NaN,12.0,NaN
1853,realme pad wifi+4g tablet | 6gb ram 128gb rom ...,Smartphones,6.0,128.0,7100.0,NaN,NaN,NaN
239,"realme narzo 50 5g (hyper blue, 6gb ram+128gb ...",Smartphones,6.0,128.0,NaN,NaN,NaN,NaN
2063,hp victus gaming latest amd ryzen 5 5600h proc...,Laptops,8.0,512.0,NaN,NaN,NaN,NaN
419,boat flash edition smart watch with activity t...,Smartwatches,NaN,NaN,NaN,1.30,NaN,NaN
1924,sony bravia 108 cm (43 inches) 4k ultra hd sma...,Televisions,NaN,NaN,NaN,43.00,NaN,NaN
1349,ptron volta fc17 25w type-c to type-c pd super...,Chargers and Cables,NaN,NaN,NaN,NaN,25.0,NaN
788,"samsung galaxy m13 (stardust brown, 6gb, 128gb...",Smartphones,6.0,128.0,6000.0,NaN,NaN,NaN
4185,belkin usb c to usb-c fast charging type c cab...,Chargers and Cables,NaN,NaN,NaN,NaN,60.0,NaN
2212,"redmi note 11 pro (stealth black, 8gb ram, 128...",Smartphones,8.0,128.0,NaN,NaN,67.0,120.0


## Phase 5 result

The exported file keeps every source record, original field, category label, and six nullable numerical specifications. Coverage is expected to vary by category and title-writing style. This is ready as an exploratory feature-engineering output; Phase 6 will use imputation inside the training pipeline and will compare whether the sparse specification features improve the target proxy.

In [5]:
df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
print("Final shape:", df.shape)
print("Original rows retained:", len(df))
print("Feature columns:", spec_columns)

Saved: electronics_feature_engineered.csv
Final shape: (5010, 24)
Original rows retained: 5010
Feature columns: ['ram_gb', 'storage_gb', 'battery_mah', 'screen_inches', 'charging_w', 'refresh_rate_hz']
